# Big Data Analytics · Session 1 — Practical
**Module INF-17 · WelfenAkademie · Business Informatics · 4th Semester**

---

Welcome to the first hands-on session! Today we will:

1. Set up our Python environment
2. Load a real-world dataset directly from Hugging Face
3. Take a first look at the data: rows, columns, and what they mean
4. Compute basic statistics: average, minimum, maximum, most frequent values
5. Create two simple charts: a **bar chart** and a **histogram**
6. Work on a short **group activity**

> 💡 **How to use this notebook:** Run each cell from top to bottom by clicking the ▶ button (or pressing `Shift + Enter`). Read the explanations in the grey text boxes before each code block.

---
## 1 · Setup — Installing and Importing Libraries

Python comes with many built-in functions, but for data analysis we rely on a few extra **libraries** (pre-written collections of tools).

| Library | What it does |
|---|---|
| `datasets` | Lets us download datasets from Hugging Face with a single line of code |
| `pandas` | The go-to library for working with tables of data |
| `matplotlib` | Creates charts and plots |

The cell below **installs** all three libraries. You only need to run this once per environment.

In [ ]:
# Install the required libraries
# The exclamation mark (!) means: run this as a terminal command, not Python
!pip install datasets pandas matplotlib --quiet

In [ ]:
# Import the libraries we will use throughout this notebook
import pandas as pd                      # pandas → we'll call it 'pd' for short
import matplotlib.pyplot as plt          # matplotlib's plotting module → 'plt'
from datasets import load_dataset        # function to fetch datasets from Hugging Face

print("✅ Libraries loaded successfully!")

---
## 2 · Loading the Dataset

We are going to work with the **Top Hits Spotify** dataset — a collection of ~2,000 popular Spotify songs from the years 1998–2020, with audio features for each track.

**Dataset source:** [huggingface.co/datasets/osanseviero/top-hits-spotify](https://huggingface.co/datasets/osanseviero/top-hits-spotify)

We load it with one line of code and convert it to a **DataFrame** — pandas' name for a table with rows and columns, similar to an Excel spreadsheet.

In [ ]:
# Load the dataset from Hugging Face
dataset = load_dataset("osanseviero/top-hits-spotify")

# The dataset has a 'train' split — convert it to a pandas DataFrame
df = dataset["train"].to_pandas()

print("✅ Dataset loaded!")

---
## 3 · First Look at the Data

Before doing any analysis, always start by **understanding what you have**. We will answer three questions:
- How many rows and columns does the dataset have?
- What are the column names and data types?
- What do the first few rows look like?

### 3.1 · Shape — How big is the dataset?

In [ ]:
# .shape returns (number of rows, number of columns)
rows, columns = df.shape
print(f"The dataset has {rows} rows and {columns} columns.")

### 3.2 · Column Names and Data Types

Each column stores a different piece of information about a song. Here is a quick reference guide:

| Column | Description |
|---|---|
| `artist` | Name of the artist |
| `song` | Song title |
| `duration_ms` | Duration of the song in milliseconds |
| `explicit` | Whether the song contains explicit content (True/False) |
| `year` | Year the song appeared in the Spotify charts |
| `popularity` | Spotify popularity score (0–100, higher = more popular) |
| `danceability` | How suitable the track is for dancing (0.0–1.0) |
| `energy` | Perceptual measure of intensity and activity (0.0–1.0) |
| `key` | Musical key the song is in (0 = C, 1 = C#, … 11 = B) |
| `loudness` | Overall loudness in decibels (typically –20 to 0 dB) |
| `mode` | Modality: 1 = major key, 0 = minor key |
| `speechiness` | Proportion of spoken words in the track (0.0–1.0) |
| `acousticness` | Confidence that the track is acoustic (0.0–1.0) |
| `instrumentalness` | Predicts whether the track has no vocals (0.0–1.0) |
| `liveness` | Probability the track was recorded live (0.0–1.0) |
| `valence` | Musical positiveness — happy/cheerful vs. sad/angry (0.0–1.0) |
| `tempo` | Estimated tempo in beats per minute (BPM) |
| `genre` | Music genre (e.g., pop, rock, hip hop) |

In [ ]:
# .dtypes shows the data type of every column
# 'object' usually means text, 'int64' = integer number, 'float64' = decimal number
print(df.dtypes)

### 3.3 · Preview — First 5 Rows

In [ ]:
# .head(n) shows the first n rows of the DataFrame
df.head(5)

In [ ]:
# .info() gives a compact overview: row count, column names, non-null counts, and data types
# 'Non-Null Count' tells us how many values are NOT missing
df.info()

---
## 4 · Basic Statistics

Now let's compute some simple numbers to get a feel for the data.

### 4.1 · Summary Statistics for Numeric Columns

`.describe()` is a quick shortcut that calculates the most important statistics for every numeric column at once.

In [ ]:
# .describe() calculates: count, mean, std (standard deviation), min, quartiles, max
# Tip: .T transposes the table — easier to read when there are many columns
df.describe().T.round(2)

### 4.2 · Digging Deeper — Individual Statistics

Let's compute a few statistics for specific columns that tell an interesting story.

In [ ]:
# --- Popularity ---
print("=== Popularity (0–100) ===")
print(f"  Average popularity : {df['popularity'].mean():.1f}")
print(f"  Minimum popularity : {df['popularity'].min()}")
print(f"  Maximum popularity : {df['popularity'].max()}")
print(f"  Median popularity  : {df['popularity'].median():.1f}")

In [ ]:
# --- Duration (convert ms → minutes for readability) ---
df['duration_min'] = df['duration_ms'] / 60_000   # add a helper column in minutes

print("=== Song Duration ===")
print(f"  Average duration : {df['duration_min'].mean():.2f} minutes")
print(f"  Shortest song    : {df['duration_min'].min():.2f} minutes")
print(f"  Longest song     : {df['duration_min'].max():.2f} minutes")

In [ ]:
# --- Most frequent values (categorical columns) ---

# Most common artist in the dataset
top_artists = df['artist'].value_counts().head(10)
print("=== Top 10 Artists by Number of Songs ===")
print(top_artists.to_string())

print()

# Year with the most songs
top_years = df['year'].value_counts().sort_index()
print(f"\nYear with the most songs: {df['year'].value_counts().idxmax()} "
      f"({df['year'].value_counts().max()} songs)")

In [ ]:
# --- Explicit content ---
explicit_counts = df['explicit'].value_counts()
explicit_pct    = df['explicit'].mean() * 100

print("=== Explicit Content ===")
print(explicit_counts.to_string())
print(f"\n{explicit_pct:.1f}% of songs in the dataset are marked as explicit.")

---
## 5 · Visualizations

Numbers alone can be hard to interpret. Charts make patterns immediately visible. We will create two types:

- **Bar chart** — good for comparing categories (e.g., how many songs per year)
- **Histogram** — good for showing the distribution of a continuous number (e.g., how popularity scores are spread out)

### 5.1 · Bar Chart — Number of Songs per Year

In [ ]:
# Count how many songs appear in each year, then sort by year
songs_per_year = df['year'].value_counts().sort_index()

# --- Create the bar chart ---
fig, ax = plt.subplots(figsize=(12, 5))   # create a figure (12 inches wide, 5 tall)

ax.bar(
    songs_per_year.index,    # x-axis: the years
    songs_per_year.values,   # y-axis: the counts
    color='steelblue',
    edgecolor='white'
)

# Labels and formatting
ax.set_title("Number of Top-Hit Songs per Year (1998–2020)", fontsize=14, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Number of Songs")
ax.set_xticks(songs_per_year.index)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()   # prevent labels from being cut off
plt.show()

print("\n💬 Discussion: Which year has the most songs? What could explain the differences?")

### 5.2 · Histogram — Distribution of Song Popularity

A **histogram** groups values into buckets (called *bins*) and shows how many songs fall into each bucket. This tells us the **shape** of the data — for example, whether most songs have a similar popularity score, or whether scores are spread out widely.

In [ ]:
# --- Create the histogram ---
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    df['popularity'],
    bins=20,              # split the range 0–100 into 20 equal buckets
    color='coral',
    edgecolor='white'
)

# Add a vertical line for the average popularity
avg_popularity = df['popularity'].mean()
ax.axvline(avg_popularity, color='darkred', linestyle='--', linewidth=1.5,
           label=f'Average: {avg_popularity:.1f}')

# Labels and formatting
ax.set_title("Distribution of Song Popularity Scores", fontsize=14, fontweight='bold')
ax.set_xlabel("Popularity Score (0–100)")
ax.set_ylabel("Number of Songs")
ax.legend()

plt.tight_layout()
plt.show()

print("\n💬 Discussion: Is the distribution symmetric or skewed? What does that tell us?")

### 5.3 · Bonus — Two Charts Side by Side

Let's compare **danceability** and **energy** in one view. These are two of Spotify's most interesting audio features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))   # 1 row, 2 columns of subplots

# Left chart: Danceability
axes[0].hist(df['danceability'], bins=20, color='mediumseagreen', edgecolor='white')
axes[0].axvline(df['danceability'].mean(), color='darkgreen', linestyle='--', linewidth=1.5,
                label=f"Avg: {df['danceability'].mean():.2f}")
axes[0].set_title("Danceability", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Danceability (0.0 – 1.0)")
axes[0].set_ylabel("Number of Songs")
axes[0].legend()

# Right chart: Energy
axes[1].hist(df['energy'], bins=20, color='mediumpurple', edgecolor='white')
axes[1].axvline(df['energy'].mean(), color='indigo', linestyle='--', linewidth=1.5,
                label=f"Avg: {df['energy'].mean():.2f}")
axes[1].set_title("Energy", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Energy (0.0 – 1.0)")
axes[1].set_ylabel("Number of Songs")
axes[1].legend()

plt.suptitle("Audio Features: Danceability vs. Energy", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💬 Discussion: How do the two distributions differ? What does a high energy score mean for a song?")

---
## 🎯 Session Recap

Great work! Here is what we covered today:

| Skill | Python code used |
|---|---|
| Load a dataset | `load_dataset(...)` → `.to_pandas()` |
| Check size | `df.shape` |
| Inspect column types | `df.dtypes`, `df.info()` |
| Preview rows | `df.head(5)` |
| Summary statistics | `df.describe()`, `.mean()`, `.min()`, `.max()` |
| Most frequent values | `df['column'].value_counts()` |
| Bar chart | `ax.bar(x, y)` |
| Histogram | `ax.hist(data, bins=20)` |

**Next session:** We will look at data quality — handling missing values, wrong data types, duplicates — and create more chart types including box plots and scatter plots.

---
## 🎨 Optional: Better Visualizations with Seaborn

**Seaborn** is a Python library built on top of matplotlib. It produces publication-quality
charts with much less code, smarter defaults, and built-in statistical overlays.

| Feature | matplotlib | seaborn |
|---|---|---|
| Setup effort | More manual | Minimal |
| Statistical overlays | Manual | Built-in (KDE, regression lines, …) |
| Multi-variable plots | Verbose | Concise |
| Style | Basic | Polished by default |

In [ ]:
!pip install seaborn --quiet

import seaborn as sns

# Set a clean global style — applies to all subsequent charts
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Seaborn ready!')

### 6.1 · Styled Histogram with KDE Overlay

A **KDE (Kernel Density Estimate)** is a smooth curve that shows the probability
distribution of the data — think of it as a smoothed-out histogram.
Seaborn can draw both at the same time with a single function call.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: matplotlib (what we used before)
axes[0].hist(df['popularity'], bins=20, color='coral', edgecolor='white')
axes[0].set_title('matplotlib — basic histogram', fontweight='bold')
axes[0].set_xlabel('Popularity')
axes[0].set_ylabel('Count')

# Right: seaborn — same data, but with a KDE curve and better styling
sns.histplot(df['popularity'], bins=20, kde=True, color='coral', ax=axes[1])
axes[1].set_title('seaborn — histogram + KDE overlay', fontweight='bold')
axes[1].set_xlabel('Popularity')

plt.suptitle('Popularity Score Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💬 The KDE curve makes the shape of the distribution much easier to read.')

### 6.2 · Box Plot — Energy by Decade

A **box plot** is excellent for comparing a numeric variable across categories.
It shows the median, the middle 50 % of values (the box), and outliers (the dots).
Here we compare the *energy* of songs across different decades.

In [ ]:
# Create a 'decade' column for grouping
df['decade'] = (df['year'] // 10 * 10).astype(str) + 's'

fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    data=df,
    x='decade',
    y='energy',
    palette='Set2',
    order=sorted(df['decade'].unique()),
    ax=ax
)

ax.set_title('Song Energy by Decade', fontsize=14, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Energy (0.0 – 1.0)')

plt.tight_layout()
plt.show()

print('💬 Has the energy of top hits changed over the decades?')

### 6.3 · Correlation Heatmap

A **correlation heatmap** shows how strongly pairs of numeric variables are related.
- A value close to **+1** means: when one goes up, the other tends to go up too
- A value close to **−1** means: when one goes up, the other tends to go down
- A value close to **0** means: no clear relationship

This is one of the most useful first steps in any data analysis — it immediately
reveals which variables are worth investigating together.

In [ ]:
# Select the numeric audio feature columns
audio_cols = ['popularity', 'danceability', 'energy', 'loudness',
              'speechiness', 'acousticness', 'liveness', 'valence', 'tempo']

corr_matrix = df[audio_cols].corr()   # calculate pairwise correlations

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,          # print the correlation value inside each cell
    fmt='.2f',           # format: 2 decimal places
    cmap='coolwarm',     # red = positive, blue = negative correlation
    center=0,            # white = 0 (no correlation)
    linewidths=0.5,
    ax=ax
)

ax.set_title('Correlation Between Audio Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💬 Which pairs of features are most strongly correlated?')
print('   Hint: look at energy vs. loudness and energy vs. acousticness.')

---
## 🖱️ Optional: Interactive Charts with Bokeh

**Bokeh** generates charts that run in the browser — users can zoom, pan, and
hover over data points to see details. This makes it a great choice for
exploration and presentations.

Unlike matplotlib and seaborn (which produce static images), Bokeh outputs
interactive HTML — the same chart can be embedded in a dashboard or a web page.

In [ ]:
!pip install bokeh --quiet

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis256

output_notebook()   # render charts inline in the notebook
print('✅ Bokeh ready!')

### 7.1 · Interactive Scatter Plot — Danceability vs. Energy

Each dot represents one song. **Hover over any dot** to see the song title,
artist, year, and popularity score. Use the toolbar on the right to zoom and pan.

In [ ]:
# Prepare a clean subset of the data for Bokeh
plot_df = df[['danceability', 'energy', 'song', 'artist', 'year', 'popularity']].dropna().copy()

source = ColumnDataSource(plot_df)

# Define what appears when hovering over a dot
hover = HoverTool(tooltips=[
    ('Song',       '@song'),
    ('Artist',     '@artist'),
    ('Year',       '@year'),
    ('Popularity', '@popularity'),
])

# Color dots by popularity (dark = less popular, bright = more popular)
mapper = linear_cmap(
    field_name='popularity',
    palette=Viridis256,
    low=plot_df['popularity'].min(),
    high=plot_df['popularity'].max()
)

p = figure(
    width=750, height=500,
    title='Danceability vs. Energy  (color = popularity)',
    x_axis_label='Danceability (0.0 – 1.0)',
    y_axis_label='Energy (0.0 – 1.0)',
    tools=[hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset'],
    toolbar_location='right'
)

p.circle(
    x='danceability', y='energy',
    source=source,
    size=7, color=mapper, alpha=0.65,
    line_color=None
)

show(p)

### 7.2 · Interactive Bar Chart — Average Popularity per Year

Hover over each bar to see the exact average popularity score for that year.

In [ ]:
from bokeh.models import ColumnDataSource

# Compute average popularity per year
avg_pop = df.groupby('year')['popularity'].mean().reset_index()
avg_pop.columns = ['year', 'avg_popularity']
avg_pop['year_str'] = avg_pop['year'].astype(str)   # Bokeh needs string categories for x-axis

source_bar = ColumnDataSource(avg_pop)

hover_bar = HoverTool(tooltips=[
    ('Year',               '@year'),
    ('Avg. Popularity', '@avg_popularity{0.1f}'),
])

p2 = figure(
    x_range=avg_pop['year_str'].tolist(),
    width=800, height=400,
    title='Average Song Popularity per Year',
    x_axis_label='Year',
    y_axis_label='Average Popularity',
    tools=[hover_bar, 'pan', 'reset'],
    toolbar_location='right'
)

p2.vbar(
    x='year_str', top='avg_popularity',
    source=source_bar,
    width=0.7, color='steelblue', alpha=0.8,
    line_color='white'
)

p2.xaxis.major_label_orientation = 0.8   # rotate x labels slightly

show(p2)

> **Note:** Bokeh charts render as interactive HTML widgets inside the notebook.
> If you want to share a chart with someone who doesn't have Python, you can
> export it as a standalone HTML file with `output_file('chart.html')` instead
> of `output_notebook()`.